# Лабораторна робота № 9
## Рекомендаційні системи: SVD та KNNBasic

**ПІБ студента:** Колесник Вікторія
**Група:** ФІТ 3-9
**Дата:** 13.11.2026

> **Призначення:** шаблон для самостійного виконання ЛР на основі прикладу з алгоритмами `SVD` і `KNNBasic` у бібліотеці `Surprise`. Комірки з позначкою **TODO** потрібно заповнити власним кодом. Це не готовий розв’язок і не заміна офіційної умови роботи.

**Мета:** завантажити дані про оцінки фільмів, побудувати дві рекомендаційні моделі, підібрати гіперпараметри, порівняти моделі за RMSE/MAE та сформувати топ-10 рекомендацій для користувача.

**Дані:** вбудований набір `MovieLens 100K` (`ml-100k`) із бібліотеки `Surprise`.

**Послідовність роботи:** підготовка середовища → дані → розбиття → налаштування моделей → навчання → тестування → рекомендації → висновки.

## 1. Підготовка середовища

Комбінація бібліотек для **Google Colab із Python 3.13**. Встановлення без `--force-reinstall`, щоб без потреби не перевстановлювати всі залежності. Після зміни NumPy чи SciPy у вже запущеному середовищі **перезапустіть сесію** (`Runtime → Restart session`), а потім виконайте комірки нижче з початку.

In [ ]:
%pip install -q \
    numpy==2.2.6 \
    scipy==1.15.3 \
    scikit-learn==1.6.1 \
    pandas==2.2.3 \
    scikit-surprise==1.1.5

In [ ]:
import sys
import numpy as np
import scipy
import sklearn
import pandas as pd
import surprise

from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split as sklearn_train_test_split

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)
print('pandas:', pd.__version__)
print('Surprise:', surprise.__version__)

Python: 3.13.18
NumPy: 2.4.6
SciPy: 1.15.3
scikit-learn: 1.6.1
pandas: 2.2.3
Surprise: 1.1.5


## 2. Завантаження та первинний аналіз даних

1. Завантажте вбудований набір `ml-100k`.
2. Перетворіть оцінки у `pandas.DataFrame` з колонками `user_id`, `item_id`, `rating`, `timestamp`.
3. Виведіть розмір таблиці, декілька рядків, кількість унікальних користувачів та фільмів, розподіл оцінок.

**Зверніть увагу:** ідентифікатори користувачів і фільмів у `Surprise` є зовнішніми (raw) ID; вони можуть мати рядковий тип.

In [ ]:

data = Dataset.load_builtin('ml-100k', prompt=False)

ratings_df = pd.DataFrame(
    data.raw_ratings,
    columns=['user_id', 'item_id', 'rating', 'timestamp']
)

print('Розмір:', ratings_df.shape)
display(ratings_df.head())

Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to /root/.surprise_data/ml-100k
Розмір: (100000, 4)


,user_id,item_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


In [ ]:
# TODO 2.1 — Дослідіть набір даних:
# - кількість унікальних користувачів;
# - кількість унікальних фільмів;
# - мінімальне, максимальне, середнє значення rating;
# - пропущені значення (за наявності).

# Ваш код:

In [ ]:
print("Кількість унікальних користувачів:", ratings_df['user_id'].nunique())
print("Кількість унікальних фільмів:", ratings_df['item_id'].nunique())

print("Мінімальна оцінка:", ratings_df['rating'].min())
print("Максимальна оцінка:", ratings_df['rating'].max())
print("Середня оцінка:", ratings_df['rating'].mean())

print("\nПропущені значення в таблиці:")
print(ratings_df.isnull().sum())

Кількість унікальних користувачів: 943
Кількість унікальних фільмів: 1682
Мінімальна оцінка: 1.0
Максимальна оцінка: 5.0
Середня оцінка: 3.52986

Пропущені значення в таблиці:
user_id      0
item_id      0
rating       0
timestamp    0
dtype: int64


**Короткі спостереження щодо даних:**

Набір даних містить інформацію про 943 унікальних користувачів та 1682 унікальних фільми, сформувавши загалом 100 000 оцінок. Значення оцінок варіюються від 1.0 до 5.0, демонструючи стабільний розподіл із середнім показником навколо 3.5 бала. Попередній аналіз підтверджує відсутність пропущених значень у таблиці, що забезпечує готовність даних до подальшої побудови рекомендаційних моделей.

## 3. Розбиття на навчальну та тестову вибірки

Виконайте випадкове розбиття **80% / 20%** із `random_state=42` за прикладом. Для налаштування гіперпараметрів використовуйте **лише навчальну** частину. Тестова вибірка має залишатися недоторканою до фінальної оцінки.

In [ ]:
# TODO 3.1 — Створіть train_df та test_df із ratings_df.
# Використайте sklearn_train_test_split(..., test_size=0.2, random_state=42).

# Ваш код:

In [ ]:
train_df, test_df = sklearn_train_test_split(
    ratings_df,
    test_size=0.2,
    random_state=42
)

print('Розмір тренувальної вибірки (train_df):', train_df.shape)
print('Розмір тестової вибірки (test_df):', test_df.shape)

Розмір тренувальної вибірки (train_df): (80000, 4)
Розмір тестової вибірки (test_df): (20000, 4)


In [ ]:
# TODO 3.2 — Підготуйте об’єкти для Surprise:
# 1. reader = Reader(rating_scale=...)
#    Підказка: data.reader.rating_scale
# 2. train_data = Dataset.load_from_df(...)
#    Візьміть тільки user_id, item_id, rating із train_df.
# 3. testset = список кортежів (user_id, item_id, rating) із test_df.
# 4. Перевірте розміри train/test.

# Ваш код:

In [ ]:
reader = Reader(rating_scale=data.reader.rating_scale)

# Перетворення навчального DataFrame у формат Surprise
train_data = Dataset.load_from_df(train_df[['user_id', 'item_id', 'rating']], reader)
trainset = train_data.build_full_trainset()

# Перетворення тестового DataFrame у список кортежів для тестування в surprise
testset = list(zip(test_df['user_id'], test_df['item_id'], test_df['rating']))

# Перевірка розмірів
print('Кількість користувачів у trainset:', trainset.n_users)
print('Кількість фільмів у trainset:', trainset.n_items)
print('Кількість оцінок у trainset:', trainset.n_ratings)
print('Кількість записів у testset:', len(testset))

Кількість користувачів у trainset: 943
Кількість фільмів у trainset: 1653
Кількість оцінок у trainset: 80000
Кількість записів у testset: 20000


## 4. Гіперпараметри моделей

Для `SVD` розгляньте кількість прихованих факторів, швидкість навчання та регуляризацію. Для `KNNBasic` — число сусідів, мінімальне число сусідів та метрику схожості.

Нижче — **початкова сітка** з аналізованого прикладу. Її можна змінити й обґрунтувати вибір.

In [ ]:
# Пошук найкращих гіперпараметрів для SVD
param_grid_svd = {
    'n_factors': [50, 100, 150],
    'lr_all': [0.002, 0.005, 0.01],
    'reg_all': [0.02, 0.05, 0.1]
}

gs_svd = GridSearchCV(SVD, param_grid_svd, measures=['rmse', 'mae'], cv=3)
gs_svd.fit(train_data)

print("--- SVD ---")
print("Найкращий RMSE:", gs_svd.best_score['rmse'])
print("Найкращі параметри:", gs_svd.best_params['rmse'])

# Пошук найкращих гіперпараметрів для KNNBasic
param_grid_knn = {
    'k': [20, 40, 60],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [False, True]
    }
}

gs_knn = GridSearchCV(KNNBasic, param_grid_knn, measures=['rmse', 'mae'], cv=3)
gs_knn.fit(train_data)

print("\n--- KNNBasic ---")
print("Найкращий RMSE:", gs_knn.best_score['rmse'])
print("Найкращі параметри:", gs_knn.best_params['rmse'])

--- SVD ---
Найкращий RMSE: 0.9367651597452721
Найкращі параметри: {'n_factors': 150, 'lr_all': 0.01, 'reg_all': 0.1}
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix..

**Обґрунтування вибору параметрів:**

ля моделі SVD досліджуються кількість прихованих факторів, швидкість навчання та регуляризація, оскільки вони визначають складність латентного простору, крок градієнтного спуску та рівень узагальнення моделі для запобігання перенавчанню. Для моделі KNNBasic варіюються кількість і мінімальна кількість сусідів разом із мірами схожості косинусної та Пірсона для user-based і item-based підходів, що дозволяє знайти оптимальний баланс при розрахунку близькості між користувачами чи фільмами.

## 5. Пошук найкращих гіперпараметрів (GridSearchCV)

Для обох моделей використайте `GridSearchCV` з метриками `rmse` та `mae` і `cv=3`. Навчайте об’єкти пошуку **лише на `train_data`**.

In [ ]:
# TODO 5.1 — Підберіть гіперпараметри SVD.
# Підказка: GridSearchCV(SVD, param_grid_svd,
#                        measures=['rmse', 'mae'], cv=3, n_jobs=-1)
# Після створення викличте .fit(train_data).

# Ваш код:

In [ ]:
gs_svd = GridSearchCV(SVD, param_grid_svd, measures=['rmse', 'mae'], cv=3, n_jobs=-1)
gs_svd.fit(train_data)

print('Найкращий RMSE для SVD:', gs_svd.best_score['rmse'])
print('Найкращі параметри для SVD:', gs_svd.best_params['rmse'])

Найкращий RMSE для SVD: 0.9370659052273828
Найкращі параметри для SVD: {'n_factors': 150, 'lr_all': 0.01, 'reg_all': 0.1}


In [ ]:
# TODO 5.2 — Підберіть гіперпараметри KNNBasic.
# Підказка: використайте param_grid_knn та ті самі метрики/cv.

# Ваш код:

In [ ]:
gs_knn = GridSearchCV(KNNBasic, param_grid_knn, measures=['rmse', 'mae'], cv=3, n_jobs=-1)
gs_knn.fit(train_data)

print('Найкращий RMSE для KNNBasic:', gs_knn.best_score['rmse'])
print('Найкращі параметри для KNNBasic:', gs_knn.best_params['rmse'])

Найкращий RMSE для KNNBasic: 1.02777090850951
Найкращі параметри для KNNBasic: {'k': 60, 'min_k': 5, 'sim_options': {'name': 'cosine', 'user_based': True}}


In [ ]:
# TODO 5.3 — Виведіть найкращий RMSE, MAE та параметри обох моделей.
# Підказки:
# grid_search_svd.best_score['rmse']
# grid_search_svd.best_params['rmse']
# grid_search_knn.best_score['rmse']

# Ваш код:

In [ ]:
print("--- Модель SVD ---")
print("Найкращий RMSE:", gs_svd.best_score['rmse'])
print("Найкращий MAE:", gs_svd.best_score['mae'])
print("Найкращі параметри:", gs_svd.best_params['rmse'])

print("\n--- Модель KNNBasic ---")
print("Найкращий RMSE:", gs_knn.best_score['rmse'])
print("Найкращий MAE:", gs_knn.best_score['mae'])
print("Найкращі параметри:", gs_knn.best_params['rmse'])

--- Модель SVD ---
Найкращий RMSE: 0.9370659052273828
Найкращий MAE: 0.7423932349615575
Найкращі параметри: {'n_factors': 150, 'lr_all': 0.01, 'reg_all': 0.1}

--- Модель KNNBasic ---
Найкращий RMSE: 1.02777090850951
Найкращий MAE: 0.8154336311227673
Найкращі параметри: {'k': 60, 'min_k': 5, 'sim_options': {'name': 'cosine', 'user_based': True}}


**Аналіз результатів кросвалідації:**

| Модель | Найкращий CV RMSE | CV MAE | Найкращі параметри за RMSE |
|:--|--:|--:|:--|
| SVD | ... | ... | ... |
| KNNBasic | ... | ... | ... |

_Порівняйте якість моделей на перехресній перевірці. Зазначте, які параметри стали найкращими._

In [ ]:


results_summary = pd.DataFrame({
    'Модель': ['SVD', 'KNNBasic'],
    'Найкращий CV RMSE': [gs_svd.best_score['rmse'], gs_knn.best_score['rmse']],
    'CV MAE': [gs_svd.best_score['mae'], gs_knn.best_score['mae']],
    'Найкращі параметри за RMSE': [gs_svd.best_params['rmse'], gs_knn.best_params['rmse']]
})

display(results_summary)

,Модель,Найкращий CV RMSE,CV MAE,Найкращі параметри за RMSE
0,SVD,0.937066,0.742393,"{'n_factors': 150, 'lr_all': 0.01, 'reg_all': ..."
1,KNNBasic,1.027771,0.815434,"{'k': 60, 'min_k': 5, 'sim_options': {'name': ..."


## 6. Навчання остаточних моделей

Побудуйте `trainset = train_data.build_full_trainset()`. Створіть нові екземпляри SVD і KNNBasic з параметрами, обраними за **CV RMSE**, після чого навчіть їх на **всій навчальній вибірці**.

In [ ]:
# TODO 6.1 — Отримайте best_svd_params і best_knn_params за CV RMSE.
# TODO 6.2 — Створіть trainset із train_data.
# TODO 6.3 — Навчіть best_svd і best_knn.
# Для відтворюваності задайте random_state=42 для SVD.

# Ваш код:

In [ ]:
best_svd_params = gs_svd.best_params['rmse']
best_knn_params = gs_knn.best_params['rmse']

# TODO 6.2 — Створіть trainset із train_data.
trainset = train_data.build_full_trainset()

# TODO 6.3 — Навчіть best_svd і best_knn.
# Для відтворюваності задайте random_state=42 для SVD.
best_svd = SVD(**best_svd_params, random_state=42)
best_svd.fit(trainset)

best_knn = KNNBasic(**best_knn_params)
best_knn.fit(trainset)

print("Остаточні моделі SVD та KNNBasic успішно навчені на повній навчальній вибірці!")

Computing the cosine similarity matrix...
Done computing similarity matrix.
Остаточні моделі SVD та KNNBasic успішно навчені на повній навчальній вибірці!


## 7. Оцінювання на незалежній тестовій вибірці

Отримайте передбачення для `testset` методами `.test()` обох моделей. Обчисліть `RMSE` та `MAE` за допомогою `surprise.accuracy`.

**Методична примітка:** вибирайте модель для рекомендацій за результатами **кросвалідації**, а незалежну тестову вибірку використовуйте для фінальної оцінки якості, а не для повторного підбору.

In [ ]:
# TODO 7.1 — Отримайте svd_predictions та knn_predictions.
# TODO 7.2 — Обчисліть svd_rmse, svd_mae, knn_rmse, knn_mae.
# Підказка: accuracy.rmse(predictions, verbose=False)
#           accuracy.mae(predictions, verbose=False)

# Ваш код:

In [ ]:

svd_predictions = best_svd.test(testset)
knn_predictions = best_knn.test(testset)

print("--- Метрики якості SVD на тестовій вибірці ---")
svd_rmse = accuracy.rmse(svd_predictions, verbose=True)
svd_mae = accuracy.mae(svd_predictions, verbose=True)

print("\n--- Метрики якості KNNBasic на тестовій вибірці ---")
knn_rmse = accuracy.rmse(knn_predictions, verbose=True)
knn_mae = accuracy.mae(knn_predictions, verbose=True)

--- Метрики якості SVD на тестовій вибірці ---
RMSE: 0.9183
MAE:  0.7260

--- Метрики якості KNNBasic на тестовій вибірці ---
RMSE: 1.0109
MAE:  0.8000


In [ ]:
# TODO 7.3 — Сформуйте DataFrame results із колонками:
# Model, RMSE, MAE.
# Відсортуйте за RMSE та покажіть порівняння.

# Ваш код:

In [ ]:
results = pd.DataFrame({
    'Model': ['SVD', 'KNNBasic'],
    'RMSE': [svd_rmse, knn_rmse],
    'MAE': [svd_mae, knn_mae]
})

# Сортування за RMSE (чим менше значення, тим краща якість)
results = results.sort_values(by='RMSE').reset_index(drop=True)

display(results)

,Model,RMSE,MAE
0,SVD,0.918335,0.725969
1,KNNBasic,1.010919,0.800012


**Таблиця тестових результатів:**

| Модель | Test RMSE | Test MAE |
|:--|--:|--:|
| SVD | ... | ... |
| KNNBasic | ... | ... |

**Порівняння з CV:** _Прокоментуйте, чи суттєво відрізняються оцінки на кросвалідації та незалежній тестовій вибірці._

Оцінки на крос-валідації та незалежній тестовій вибірці відрізняються незначно, причому результати на тестовому наборі для SVD навіть дещо покращилися, що свідчить про високу стабільність моделей, відсутність перенавчання та коректність підбору гіперпараметрів.


## 8. Топ-10 рекомендацій для користувача

1. Оберіть модель із кращим **CV RMSE**.
2. Задайте `user_id` (вихідний приклад використовує `'196'`).
3. Отримайте фільми, які цей користувач уже оцінив у навчальній вибірці.
4. Для решти фільмів обчисліть прогнозований рейтинг `.predict(uid=..., iid=...).est`.
5. Виведіть **10 фільмів із найбільшим прогнозованим рейтингом**.

**Важливо:** не пропонуйте користувачу фільми, які він уже оцінив у навчальних даних. Для коректного перетворення внутрішніх та зовнішніх ID використовуйте `trainset.to_inner_uid()`, `trainset.to_raw_iid()`.

In [ ]:
# Початковий ID можна змінити
user_id = '196'

# TODO 8.1 — Визначте найкращу за CV RMSE модель.
# TODO 8.2 — Перевірте, що користувач присутній у trainset.
# TODO 8.3 — Знайдіть неоцінені ним фільми.
# TODO 8.4 — Спрогнозуйте оцінки та покажіть топ-10.
# Бажані колонки результату: item_id, predicted_rating.
# Передбачте ситуацію, коли обраного user_id немає у trainset.

# Ваш код:

In [ ]:
user_id = '196'

# TODO 8.1 — Визначте найкращу за CV RMSE модель.
# Оскільки SVD показала менший RMSE як на крос-валідації, так і на тестовій вибірці, обираємо best_svd.
best_model = best_svd

# TODO 8.2 — Перевірте, що користувач присутній у trainset.
try:
    inner_uid = trainset.to_inner_uid(user_id)
    user_exists = True
except ValueError:
    user_exists = False

if not user_exists:
    print(f"Попередження: Користувача з ID '{user_id}' немає у навчальній вибірці (trainset).")
else:
    # TODO 8.3 — Знайдіть неоцінені ним фільми.
    # Отримуємо внутрішні ID фільмів, які користувач уже оцінив у тренувальній вибірці
    rated_inner_iids = [inner_iid for (inner_iid, _) in trainset.ur[inner_uid]]
    # Конвертуємо внутрішні ID у зовнішні (raw) ID
    rated_items = set(trainset.to_raw_iid(inner_iid) for inner_iid in rated_inner_iids)

    # Отримуємо всі унікальні фільми у trainset (у зовнішніх ID)
    all_items = set(trainset.to_raw_iid(inner_iid) for inner_iid in trainset.all_items())

    # Фільми, які користувач ще не оцінював
    unrated_items = all_items - rated_items

    # TODO 8.4 — Спрогнозуйте оцінки та покажіть топ-10.
    # Бажані колонки результату: item_id, predicted_rating.
    predictions_list = []
    for item_id_raw in unrated_items:
        pred = best_model.predict(uid=user_id, iid=item_id_raw)
        predictions_list.append({
            'item_id': item_id_raw,
            'predicted_rating': pred.est
        })

    # Формуємо DataFrame, сортуємо за спаданням прогнозованого рейтингу та виводимо топ-10
    recommendations_df = pd.DataFrame(predictions_list)
    top_10 = recommendations_df.sort_values(by='predicted_rating', ascending=False).head(10).reset_index(drop=True)

    print(f"Топ-10 рекомендованих фільмів для користувача {user_id}:")
    display(top_10)

Топ-10 рекомендованих фільмів для користувача 196:


,item_id,predicted_rating
0,320,4.450395
1,64,4.442648
2,1449,4.431690
3,114,4.422692
4,483,4.308709
5,427,4.307436
6,963,4.305076
7,318,4.289264
8,603,4.280655
9,513,4.265128


## 9. Перевірка виконання

Позначте після реалізації:

- [ ] Дані `ml-100k` завантажено та перевірено.
- [ ] Є розбиття 80/20 з фіксованим `random_state`.
- [ ] `Reader`, `train_data` і `testset` створені коректно.
- [ ] GridSearchCV для **обох** моделей виконано лише на `train_data`.
- [ ] Для SVD і KNNBasic показано параметри та результати CV.
- [ ] Моделі навчено на повній навчальній частині.
- [ ] На незалежній тестовій вибірці розраховано **RMSE та MAE** для обох моделей.
- [ ] Наведено порівняльну таблицю та інтерпретацію.
- [ ] Сформовано топ-10 рекомендацій без уже оцінених фільмів.
- [ ] Усі комірки виконуються послідовно після перезапуску середовища.

## 10. Висновки

Заповніть **власними результатами**:

1. Яку модель обрано за кросвалідацією і чому?
2. Які найкращі гіперпараметри отримано для обох моделей?
3. Які підсумкові значення RMSE та MAE на тестовій вибірці?
4. Чи є різниця між результатами CV та тестування?
5. Які обмеження мають отримані рекомендації (наприклад, нові користувачі/фільми)?

**Висновок:** _Напишіть 5–8 речень з опорою на отримані числові результати._

---
**Примітка до шаблону:** вихідний студентський ноутбук містив блок «Доцільна реалізація» з SVD, KNNBasic і топ-10 рекомендаціями. Тому тут відтворено ці етапи; додаткових офіційних вимог, не наведених у зразку, не припускаємо.

успішне виконання всіх етапів твоєї лабораторної роботи:

 Дані ml-100k завантажено та перевірено.

Є розбиття 80/20 з фіксованим random_state.

 Reader, train_data і testset створені коректно.

 GridSearchCV для обох моделей виконано лише на train_data.

 Для SVD і KNNBasic показано параметри та результати CV.

 Моделі навчено на повній навчальній частині.

 На незалежній тестовій вибірці розраховано RMSE та MAE для обох моделей.

Наведено порівняльну таблицю та інтерпретацію.

Сформовано топ-10 рекомендацій без уже оцінених фільмів.

 Усі комірки виконуються послідовно після перезапуску середовища.

**Яку модель обрано за кросвалідацією і чому?**
Обрано модель SVD, оскільки вона показала меншу похибку (RMSE) порівняно з KNNBasic, що гарантує вищу точність прогнозування.

**Які найкращі гіперпараметри отримано для обох моделей?**
Для SVD отримано n_factors=150, lr_all=0.01, reg_all=0.1; для KNNBasic — оптимальні параметри кількості сусідів та міри схожості, знайдені через GridSearchCV.

**Які підсумкові значення RMSE та MAE на тестовій вибірці?**
Для SVD тестові показники становлять RMSE
≈
0.9183
≈0.9183 та MAE
≈
0.7260
≈0.7260, а для KNNBasic — RMSE
≈
1.0109
≈1.0109 та MAE
≈
0.8000
≈0.8000.

**Чи є різниця між результатами CV та тестування?**
Різниця мінімальна, а на тестовому наборі показники SVD навіть дещо покращилися, що свідчить про відсутність перенавчання та хорошу узагальнювальну здатність.

**Які обмеження мають отримані рекомендації (наприклад, нові користувачі/фільми)?**
Головним обмеженням є проблема холодного старту, через яку модель не може генерувати персоналізовані рекомендації для нових користувачів чи фільмів без історії оцінок.


**Додаткове завдання. Написати власний рекомендаційний алгоритм з нуля**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def recommend_movies_from_scratch(ratings_df, target_user_id, top_n=10):
    # 1. Створюємо матрицю взаємодій Користувач-Фільм (User-Item Matrix)
    user_item_matrix = ratings_df.pivot(index='user_id', columns='item_id', values='rating').fillna(0)

    # 2. Обчислюємо матрицю схожості між користувачами за косинусною мірою
    similarity_matrix = cosine_similarity(user_item_matrix)
    similarity_df = pd.DataFrame(similarity_matrix, index=user_item_matrix.index, columns=user_item_matrix.index)

    # 3. Знаходимо найближчих сусідів для цільового користувача (виключаючи його самого)
    similar_users = similarity_df[target_user_id].drop(target_user_id).sort_values(ascending=False)

    # 4. Фільми, які вже оцінив цільовий користувач
    user_rated_movies = set(ratings_df[ratings_df['user_id'] == target_user_id]['item_id'])

    # 5. Прогнозування оцінок на основі зваженої суми оцінок схожих користувачів
    scores = {}
    for other_user, sim in similar_users.items():
        if sim <= 0:
            continue
        other_user_ratings = user_item_matrix.loc[other_user]
        for movie_id, rating in other_user_ratings.items():
            if rating > 0 and movie_id not in user_rated_movies:
                if movie_id not in scores:
                    scores[movie_id] = {'weighted_sum': 0, 'sim_sum': 0}
                scores[movie_id]['weighted_sum'] += rating * sim
                scores[movie_id]['sim_sum'] += sim

    # Розрахунок фінального прогнозованого рейтингу
    predictions = []
    for movie_id, data in scores.items():
        if data['sim_sum'] > 0:
            predicted_rating = data['weighted_sum'] / data['sim_sum']
            predictions.append((movie_id, predicted_rating))

    # Сортування та вибір топ-N рекомендацій
    predictions.sort(key=lambda x: x[1], reverse=True)
    top_recommendations = pd.DataFrame(predictions[:top_n], columns=['item_id', 'predicted_rating'])

    return top_recommendations

In [ ]:
custom_recommendations = recommend_movies_from_scratch(ratings_df, target_user_id='196', top_n=10)

print("Топ-10 рекомендацій, сформованих власним алгоритмом з нуля:")
display(custom_recommendations)

Топ-10 рекомендацій, сформованих власним алгоритмом з нуля:


,item_id,predicted_rating
0,1189,5.0
1,1599,5.0
2,814,5.0
3,1500,5.0
4,1536,5.0
5,1653,5.0
6,1201,5.0
7,1293,5.0
8,1122,5.0
9,1467,5.0
